# Laboratori 10: Integrimi Monte Carlo

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si shndërrohet një integral në një mesatare statistike dhe si vlerësohet gabimi pa e njohur përgjigjen?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- rishkruajë integralin si pritje;
- nxjerrë vlerësuesin dhe gabimin standard;
- shpjegojë ligjin $N^{-1/2}$ dhe koston $\varepsilon^{-2}$;
- vlerësojë $\pi$ dhe integrale një- e shumëpërmasore;
- përdorë çifte antitetike dhe variabla kontrolli;
- llogarisë mesataren dhe variancën në mënyrë rrjedhëse;
- interpretojë intervalin e besimit pa e ngatërruar me një kufi determinist.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Monte Carlo nuk prodhon një përgjigje të vetme të pandryshueshme. Produkti i llogaritjes është çifti “vlerësim + gabim statistikor”, së bashku me farën, numrin e mostrave dhe metodën e kampionimit. Një numër me shumë shifra pa gabim standard është raportim i paplotë.

Ligji $N^{-1/2}$ është njëkohësisht avantazh dhe kufizim. Ai nuk përkeqësohet drejtpërdrejt me dimensionin, por është i ngadaltë: një shifër dhjetore shtesë kërkon afërsisht njëqind herë më shumë mostra. Reduktimi i variancës synon konstanten përpara $N^{-1/2}$.


## 1. Integrali si pritje

Për $U\sim U(a,b)$,

$$\mathbb E[f(U)]=\frac{1}{b-a}\int_a^b f(x)\,dx.$$

Prandaj

$$I=\int_a^b f(x)\,dx=(b-a)\mathbb E[f(U)].$$

Me $N$ mostra të pavarura $U_i$ përdorim

$$\widehat I_N=(b-a)\frac1N\sum_{i=1}^{N}f(U_i).$$

Ky vlerësues është i paanshëm kur pritja ekziston. Ai është rastësor: dy fara të ndryshme japin vlera të ndryshme, por me të njëjtën shpërndarje teorike.


## 2. Shembull me dorë: $\int_0^1x^2dx$

Për $U\sim U(0,1)$, $I=\mathbb E[U^2]=1/3$. Varianca e një kontributi është

$$\operatorname{Var}(U^2)=\mathbb E[U^4]-\mathbb E[U^2]^2
=\frac15-\frac19=\frac4{45}.$$

Prandaj

$$\operatorname{SE}(\widehat I_N)=\sqrt{\frac{4}{45N}}.
$$

Për $N=1000$, $SE\approx9.43\times10^{-3}$. Ky parashikim bëhet para simulimit.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


In [ ]:
def mc_uniform(f,a,b,N,rng=rng):
    u=rng.uniform(a,b,N)
    y=(b-a)*f(u)
    vleresimi=y.mean()
    se=y.std(ddof=1)/np.sqrt(N)
    return vleresimi,se

for N in [100,1000,10_000,100_000]:
    I,se=mc_uniform(lambda x:x*x,0,1,N)
    print(f'N={N:6d}: I={I:.6f}, SE={se:.3e}, z={(I-1/3)/se:+.2f}')


## 3. Pse gabimi bie si $N^{-1/2}$

Për $Y_i=(b-a)f(U_i)$ të pavarura me variancë $\sigma_Y^2$,

$$\operatorname{Var}\left(\frac1N\sum_iY_i\right)
=\frac1{N^2}\sum_i\operatorname{Var}(Y_i)
=\frac{\sigma_Y^2}{N}.$$

Pra devijimi standard i vlerësuesit është $\sigma_Y/\sqrt N$. Për ta ulur gabimin me faktor $q$, numri i mostrave duhet rritur me $q^2$. Kjo konvergjencë është e ngadaltë, por pothuajse e pavarur nga dimensioni; kjo është arsyeja kryesore e Monte Carlo-s në dimensione të larta.


In [ ]:
Ns=2**np.arange(5,19); gab=[]; se_lista=[]
for N in Ns:
    I,se=mc_uniform(lambda x:x*x,0,1,int(N))
    gab.append(abs(I-1/3)); se_lista.append(se)
fig,ax=plt.subplots(figsize=(7,4))
ax.loglog(Ns,gab,'o',label='gabimi i një realizimi')
ax.loglog(Ns,se_lista,'-',label='SE i vlerësuar')
ax.loglog(Ns,se_lista[0]*np.sqrt(Ns[0]/Ns),'--',label='$N^{-1/2}$')
ax.set(xlabel='N',ylabel='gabimi'); ax.grid(True,which='both',alpha=.3); ax.legend(); plt.show()


## 4. Shembulli gjeometrik i $\pi$-së

Në katrorin $[-1,1]^2$, disku njësi zë pjesën $\pi/4$ të sipërfaqes. Për treguesin

$$H_i=\mathbf1(X_i^2+Y_i^2\leq1),$$

vlerësuesi është

$$\widehat\pi=4\bar H.$$

Meqë $H$ është Bernoulli me $p=\pi/4$,

$$\operatorname{SE}(\widehat\pi)=4\sqrt{\frac{p(1-p)}{N}}.$$

Në praktikë zëvendësojmë $p$ me $\bar H$.


In [ ]:
def pi_mc(N,rng=rng):
    xy=rng.uniform(-1,1,size=(N,2))
    h=np.sum(xy*xy,axis=1)<=1
    p=h.mean(); return 4*p,4*np.sqrt(p*(1-p)/N),xy,h

pi_hat,se,xy,h=pi_mc(20_000)
print('pi=',pi_hat,'SE=',se,'z=',(pi_hat-np.pi)/se)
fig,ax=plt.subplots(figsize=(5,5))
ax.scatter(xy[~h,0],xy[~h,1],s=2,alpha=.25,label='jashtë')
ax.scatter(xy[h,0],xy[h,1],s=2,alpha=.25,label='brenda')
ax.set_aspect('equal'); ax.legend(markerscale=4); ax.grid(alpha=.2); plt.show()


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 5.1 Kostoja ndaj numrit të mostrave dhe tolerancës

Nëse një vlerësim i $f$ kushton $C_f$, atëherë

$$T(N)=N(C_f+c)+c_0\in\Theta(NC_f).$$

Kujtesa është $\Theta(N)$ kur mostrat ruhen, por mund të jetë $\Theta(1)$ me përditësim rrjedhës. Për një gabim standard të synuar $\varepsilon$,

$$N\approx\frac{\sigma_Y^2}{\varepsilon^2},$$

prandaj

$$T(\varepsilon)=\Theta(\varepsilon^{-2}C_f).$$

Ky derivim shpjegon pse reduktimi i variancës mund të jetë më efektiv se optimizimi i vogël i kodit.


In [ ]:
Ns=np.array([10_000,20_000,40_000,80_000,160_000,320_000]); ts=[]
for N in Ns:
    t0=perf_counter(); mc_uniform(np.sin,0,np.pi,int(N)); ts.append(perf_counter()-t0)
print('pjerrësia e kohës=',np.polyfit(np.log(Ns),np.log(ts),1)[0])


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 6.1 Mesatarja dhe varianca rrjedhëse

Për simulime shumë të mëdha nuk duhen ruajtur të gjitha mostrat. Algoritmi i Welford-it përditëson $\mu_n$ dhe $M_{2,n}$ me kujtesë konstante dhe shmang zbritjen e dy numrave të mëdhenj në formulën $E[Y^2]-E[Y]^2$. Kjo është një çështje stabiliteti numerik, jo vetëm kursim kujtese.


In [ ]:
def mc_rrjedhes(f,a,b,N,bllok=10_000,rng=rng):
    n=0; mean=0.0; M2=0.0
    for fillim in range(0,N,bllok):
        m=min(bllok,N-fillim)
        y=(b-a)*f(rng.uniform(a,b,m))
        for yi in y:
            n+=1; delta=yi-mean; mean+=delta/n; M2+=delta*(yi-mean)
    return mean,np.sqrt((M2/(n-1))/n)

I,se=mc_rrjedhes(lambda x:np.exp(-x*x),0,2,200_000)
print('I=',I,'SE=',se,'95% afërsisht=',(I-1.96*se,I+1.96*se))


## 7. Reduktimi i variancës me çifte antitetike

Për $U\sim U(0,1)$, edhe $1-U$ është uniforme. Për një funksion monoton, $f(U)$ dhe $f(1-U)$ priren të jenë të korreluara negativisht. Përdorim

$$Y=\frac{f(U)+f(1-U)}{2}.$$

Kostoja e dy vlerësimeve duhet krahasuar me uljen e variancës. Metoda nuk garanton përmirësim për çdo funksion.


In [ ]:
N=100_000
u=rng.random(N)
y_std=np.exp(u)
y_anti=.5*(np.exp(u)+np.exp(1-u))
print('integrali i saktë=',np.e-1)
print('SE standard =',y_std.std(ddof=1)/np.sqrt(N))
print('SE antitetik=',y_anti.std(ddof=1)/np.sqrt(N))
print('faktori i variancës=',y_std.var(ddof=1)/y_anti.var(ddof=1))


## 8. Variabla kontrolli

Nëse njohim $\mathbb E[G]=\mu_G$, ndërtojmë

$$Y_c=F-c(G-\mu_G).$$

Varianca minimizohet për

$$c_\star=\frac{\operatorname{Cov}(F,G)}{\operatorname{Var}(G)}.$$

Për $F=e^U$ mund të përdorim $G=U$ me $\mathbb E[U]=1/2$. Vlerësimi i $c_\star$ nga të njëjtat të dhëna shton një hollësi të vogël statistike; për laboratorin përdorim një mostër pilot.


In [ ]:
u=rng.random(200_000); F=np.exp(u); G=u
cstar=np.cov(F,G,ddof=1)[0,1]/np.var(G,ddof=1)
Yc=F-cstar*(G-.5)
print('c*=',cstar,'SE standard=',F.std(ddof=1)/np.sqrt(len(F)),
      'SE kontroll=',Yc.std(ddof=1)/np.sqrt(len(Yc)))


## 9. Shembulli i plotë dhe punishte dinamike

Vlerësoni $\pi$ dhe paraqitni vlerësimin kumulativ. Ndryshoni farën ose $N$, por jo të dyja në të njëjtin krahasim. Grafiku dinamik nuk është vetëm dekor: ai tregon se brezi i luhatjeve ngushtohet ngadalë.


In [ ]:
def pi_dinamik(N=50_000,bllok=500,DINAMIK=False):
    brenda=0; total=0; Ns=[]; v=[]; fig,ax=plt.subplots(figsize=(7,4))
    for _ in range(N//bllok):
        xy=rng.uniform(-1,1,(bllok,2)); brenda+=np.count_nonzero(np.sum(xy*xy,axis=1)<=1); total+=bllok
        Ns.append(total); v.append(4*brenda/total)
        if DINAMIK:
            ax.clear(); ax.semilogx(Ns,v); ax.axhline(np.pi,color='k',ls='--'); ax.set_ylim(2.8,3.5); plt.pause(.03)
    if not DINAMIK:
        ax.semilogx(Ns,v,label=r'$\hat\pi_N$'); ax.axhline(np.pi,color='k',ls='--',label=r'$\pi$')
    ax.set(xlabel='N',ylabel='vlerësimi'); ax.grid(alpha=.3); ax.legend(); plt.show()
    return v[-1]

print('pi final=',pi_dinamik(DINAMIK=False))


### Intervali i besimit dhe rezultati i standardizuar

Nën CLT, madhësia

$$Z=\frac{\widehat I-I}{\widehat{SE}}$$

është afërsisht normale standarde. Në eksperimente të përsëritura, rreth 95% e vlerave duhet të kenë $|Z|\lesssim1.96$. Ky kontroll kërkon që vlera e saktë të njihet dhe përdoret për verifikim të laboratorit, jo në një problem real të panjohur.


In [ ]:
z=[]
for _ in range(2000):
    I,se=mc_uniform(lambda x:x*x,0,1,1000); z.append((I-1/3)/se)
z=np.array(z)
print('mean(z)=',z.mean(),'std(z)=',z.std(ddof=1),'coverage=',np.mean(np.abs(z)<=1.96))


### Kampionimi i rëndësisë

Për një dendësi propozimi $q(x)>0$ aty ku $f(x)$ kontribuon,

$$I=\int f(x)dx=\mathbb E_q\left[\frac{f(X)}{q(X)}\right].$$

Varianca ulet kur $q$ vendos më shumë mostra në zonat ku $|f|$ është e madhe. Një $q$ i dobët mund ta rrisë variancën ose ta bëjë atë të pafundme. Peshat duhet monitoruar; disa pesha tepër të mëdha tregojnë mbulim të dobët.


In [ ]:
# I = integral_0^inf exp(-x) dx = 1; krahasojmë q_lambda eksponenciale.
N=100000
for lam in [.5,1.,2.]:
    x=rng.exponential(1/lam,N); q=lam*np.exp(-lam*x); w=np.exp(-x)/q
    print(f'lambda={lam}: I={w.mean():.6f}, SE={w.std(ddof=1)/np.sqrt(N):.3e}, maxw={w.max():.2e}')


### Dimensioni dhe kostoja

Kuadraturat në rrjetë me $n$ pika për drejtim kërkojnë $n^d$ pika në $d$ përmasa. Monte Carlo kërkon $N$ mostra pavarësisht nga $d$, megjithëse varianca e integrandit mund të ndryshojë. Përparësia shfaqet veçanërisht kur $d$ është i madh dhe kërkohet saktësi mesatare, jo shumë e lartë.


In [ ]:
# Integral i prodhimit prod cos(pi x_i/2) në [0,1]^d; vlera e saktë (2/pi)^d.
for d in [1,2,5,10]:
    X=rng.random((100000,d)); y=np.prod(np.cos(np.pi*X/2),axis=1)
    print(d,y.mean(),y.std(ddof=1)/np.sqrt(len(y)),(2/np.pi)**d)


### Pyetje për vetëkontroll

1. Pse katërfishimi i $N$ vetëm përgjysmon gabimin standard?
2. Si duhet raportuar një vlerësim Monte Carlo?
3. Kur çiftet antitetike nuk ndihmojnë?
4. Çfarë tregojnë peshat shumë të mëdha në kampionimin e rëndësisë?


# Pjesa II — GNU Octave

## 10. Integrali dhe gabimi standard

```octave
rand('seed',2026); N=100000; u=rand(N,1); y=u.^2;
I=mean(y); SE=std(y)/sqrt(N);
fprintf('I=%.8f, SE=%.3e, sakte=%.8f\n',I,SE,1/3);

xy=2*rand(N,2)-1; h=sum(xy.^2,2)<=1;
pihat=4*mean(h); SEpi=4*sqrt(mean(h)*(1-mean(h))/N);
fprintf('pi=%.8f, SE=%.3e\n',pihat,SEpi);
```

## 11. Çiftet antitetike

```octave
u=rand(N,1); y=exp(u); ya=0.5*(exp(u)+exp(1-u));
fprintf('SE standard %.3e, antitetik %.3e\n',std(y)/sqrt(N),std(ya)/sqrt(N));
```


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = 'import numpy as np\nrng=np.random.default_rng(2026)\nfor N in [1000,10000,100000]:\n u=rng.random(N); y=u*u\n print(N,y.mean(),y.std(ddof=1)/np.sqrt(N))\n'
script_path = script_dir / 'laboratori_10_integrimi_monte_carlo.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_10_integrimi_monte_carlo.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Nxora variancën e vlerësuesit.
- [ ] Lidha tolerancën me $N\sim\varepsilon^{-2}$.
- [ ] Raportova vlerësimin bashkë me SE.
- [ ] Mata përfitimin e reduktimit të variancës.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
